# 04 - Vectorized Backtesting & Factor Evaluation Showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/04_backtest_and_alpha_eval.ipynb)

In this interactive tutorial, we explore **polars-anta**'s loop-free **Vectorized Backtester** and **Alphalens-Style Factor Evaluation Suite**:
1. **1-Line Entry/Exit Crossover & Mean-Reversion Backtesting** using `.crosses_above()`, `.crosses_below()`, and `entries`/`exits` state propagation.
2. **Formatted Institutional Tearsheets (`bt.report()`) & Multi-Strategy Comparison (`pa.compare_strategies()`)** with automatic Buy & Hold benchmark tracking, transaction fees, and slippage deduction.
3. **50-Ticker Cross-Sectional Alpha Pipeline**: Spearman Rank IC / ICIR, 5-Quantile Monotonicity analysis, and Dollar-Neutral Long-Short vs. Long-Only portfolio backtesting.

In [ ]:
# Install polars-anta from GitHub (for latest backtest features in Colab)
!pip install -q git+https://github.com/huuan26/polars-anta.git matplotlib

In [ ]:
import time
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta as pa

print(f"Polars Version      : {pl.__version__}")
print(f"polars-anta Version : {pa.__version__}")

## 1. Single-Asset Strategy Backtesting (`entries` & `exits` + Crossovers)

With `polars-anta`, you don't need manual loops to hold positions between entry and exit triggers. Simply pass `entries=` and `exits=` directly into `df.anta.backtest()`:

In [ ]:
# Generate 1,500 daily OHLCV bars with trend + cyclical waves
rng = np.random.default_rng(42)
n_bars = 1_500
t = np.arange(n_bars, dtype=np.float64)
ret = 0.0004 + 0.004 * np.sin(2.0 * np.pi * t / 90.0) + rng.normal(0.0, 0.014, size=n_bars)
close = 100.0 * np.cumprod(1.0 + ret)
open_ = np.roll(close, 1) * (1.0 + rng.normal(0.0, 0.002, size=n_bars))
open_[0] = 100.0
spread = np.abs(rng.normal(0.008, 0.004, size=n_bars)) * close

df = pl.DataFrame(
    {
        "timestamp": [datetime(2020, 1, 1) + timedelta(days=int(i)) for i in range(n_bars)],
        "open": open_,
        "high": np.maximum(open_, close) + spread,
        "low": np.minimum(open_, close) - spread,
        "close": close,
        "volume": rng.lognormal(mean=14.0, sigma=0.5, size=n_bars),
    }
)

# Compute EMA(12, 26), RSI(14), and SuperTrend(10, 2.5)
t0 = time.perf_counter()
df_ta = (
    df.anta.ema(period=12, alias="ema_12")
    .anta.ema(period=26, alias="ema_26")
    .anta.rsi(period=14, alias="rsi_14")
    .anta.supertrend(period=10, multiplier=2.5, unnest=True)
)

# Strategy 1: EMA(12/26) Crossover
bt_ema = df_ta.anta.backtest(
    entries=pl.col("ema_12").anta.crosses_above("ema_26"),
    exits=pl.col("ema_12").anta.crosses_below("ema_26"),
    fee_rate=0.0005,
    slippage_rate=0.0002,
    initial_capital=100_000.0,
)

# Strategy 2: RSI(14) Mean Reversion (Buy < 30, Exit > 65)
bt_rsi = df_ta.anta.backtest(
    entries=pl.col("rsi_14").anta.crosses_below(30),
    exits=pl.col("rsi_14").anta.crosses_above(65),
    fee_rate=0.0005,
    slippage_rate=0.0002,
    initial_capital=100_000.0,
)

# Strategy 3: SuperTrend(10, 2.5) Regime Follower
bt_st = df_ta.anta.backtest(
    signal=pl.col("direction"),
    fee_rate=0.0005,
    slippage_rate=0.0002,
    initial_capital=100_000.0,
)
elapsed_ms = (time.perf_counter() - t0) * 1000.0
print(f"Computed 4 indicators + 3 full backtests in {elapsed_ms:.2f} ms!")
print(bt_st.report())

## 2. Side-by-Side Strategy Comparison & Equity Curve Visualization

In [ ]:
comparison = pa.compare_strategies(
    {
        "EMA(12/26) Crossover": bt_ema,
        "RSI(14) Mean Reversion": bt_rsi,
        "SuperTrend(10, 2.5) L/S": bt_st,
    }
).select(
    [
        "strategy",
        (pl.col("total_return") * 100).round(2).alias("total_ret_%"),
        (pl.col("benchmark_return") * 100).round(2).alias("bench_ret_%"),
        (pl.col("excess_return") * 100).round(2).alias("excess_ret_%"),
        pl.col("sharpe_ratio").round(2),
        pl.col("sortino_ratio").round(2),
        (pl.col("max_drawdown") * 100).round(2).alias("max_dd_%"),
        (pl.col("win_rate") * 100).round(1).alias("win_rate_%"),
        pl.col("n_trades").cast(pl.Int64),
    ]
)
print(comparison)

# Plot Equity Curves & Underwater Drawdown
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.5, 1]}
)

dates = bt_ema.data["timestamp"]
ax1.plot(
    dates, bt_st.data["equity_curve"], label="SuperTrend(10, 2.5) L/S", lw=1.8, color="#2ca02c"
)
ax1.plot(
    dates, bt_rsi.data["equity_curve"], label="RSI(14) Mean Reversion", lw=1.5, color="#1f77b4"
)
ax1.plot(dates, bt_ema.data["equity_curve"], label="EMA(12/26) Crossover", lw=1.5, color="#ff7f0e")
ax1.plot(
    dates,
    bt_ema.data["benchmark_equity"],
    label="Buy & Hold Benchmark",
    lw=1.2,
    ls="--",
    color="gray",
)
ax1.set_title(
    "Strategy Equity Curves vs. Buy & Hold Benchmark ($100k Initial Capital)", fontweight="bold"
)
ax1.set_ylabel("Portfolio Equity ($)")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2.fill_between(
    dates, bt_st.data["drawdown"] * 100, 0, color="#2ca02c", alpha=0.25, label="SuperTrend DD%"
)
ax2.fill_between(
    dates, bt_rsi.data["drawdown"] * 100, 0, color="#1f77b4", alpha=0.25, label="RSI DD%"
)
ax2.set_ylabel("Drawdown (%)")
ax2.set_xlabel("Date")
ax2.legend(loc="lower left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Multi-Asset Cross-Sectional Factor Evaluation & Portfolio Backtest

Evaluate a cross-sectional alpha signal across **50 stocks (25,000 rows)** using Spearman **Rank IC**, **5-Quantile Monotonicity**, and **Dollar-Neutral (`weight_scheme='dollar_neutral'`)** vs. **Long-Only (`weight_scheme='long_only'`)** portfolio backtesting:

In [ ]:
# Simulate a 50-ticker x 500-day panel (25,000 rows)
rng_panel = np.random.default_rng(7)
n_tickers, n_days = 50, 500
panel_dates = [datetime(2024, 1, 1) + timedelta(days=int(i)) for i in range(n_days)]

frames = []
for idx in range(n_tickers):
    latent_alpha = rng_panel.normal(0.0, 1.0, size=n_days)
    ret = 0.0003 + 0.0035 * np.roll(latent_alpha, 1) + rng_panel.normal(0.0, 0.015, size=n_days)
    ret[0] = 0.0
    c = (50.0 + idx) * np.cumprod(1.0 + ret)
    o = c * (1.0 - 0.002 * latent_alpha)
    frames.append(
        pl.DataFrame(
            {
                "timestamp": panel_dates,
                "symbol": [f"TICK_{idx + 1:02d}"] * n_days,
                "open": o,
                "high": np.maximum(o, c) * 1.006,
                "low": np.minimum(o, c) * 0.994,
                "close": c,
                "volume": rng_panel.lognormal(13.5, 0.4, size=n_days),
                "alpha_signal": latent_alpha,
            }
        )
    )

panel_df = pl.concat(frames).anta.forward_returns(periods=(1, 5), group_by="symbol")

# 1. Rank IC Summary
ic_stats = panel_df.anta.ic_summary(
    "alpha_signal", "fwd_ret_1", group_by="timestamp", method="spearman"
)
print("=== Rank IC Summary ===")
for k, v in ic_stats.items():
    print(f"  {k:<15}: {v:.4f}")

# 2. 5-Quantile Monotonicity Analysis
q_df = panel_df.anta.quantile_analysis(
    "alpha_signal", "fwd_ret_1", quantiles=5, group_by="timestamp"
)
print("\n=== 5-Quantile Monotonicity Table ===")
print(q_df)

# 3. Multi-Asset Portfolio Backtest (Dollar-Neutral L/S vs. Long-Only)
bt_dn = panel_df.anta.backtest(
    signal="alpha_signal",
    group_by="symbol",
    time_col="timestamp",
    weight_scheme="dollar_neutral",
    fee_rate=0.0003,
    slippage_rate=0.0002,
    initial_capital=1_000_000.0,
)
bt_lo = panel_df.anta.backtest(
    signal="alpha_signal",
    group_by="symbol",
    time_col="timestamp",
    weight_scheme="long_only",
    fee_rate=0.0003,
    slippage_rate=0.0002,
    initial_capital=1_000_000.0,
)

print("\n=== Multi-Asset Portfolio Tearsheet (Dollar-Neutral L/S) ===")
print(bt_dn.report())